[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeiraDavidson/compute_module1/blob/main/notebooks/lab_03_the_visible_staircase.ipynb)

# Lab 3: The Visible Staircase

*From* **A First Taste of Algorithms — and the Machines That Learn** *(The Curious Family Library). A computational lab: run the cells in order. A few labs end with a deliberately broken “Break it” cell — that one is meant to fail, and seeing how is the point.*

*Accompanies Chapter 3. The chapter legitimized a strange object — a procedure that calls itself — and read its cost off a tree. In this lab the strange object runs on a real machine, the hidden staircase of calls becomes visible on the screen, and the n log₂ n bill gets audited.*

## 1. A first self-call

Before towers, the smallest possible recursion — a countdown:

In [1]:
def countdown(n):
    if n == 0:
        print("liftoff")
        return
    print(n)
    countdown(n - 1)

countdown(5)

5
4
3
2
1
liftoff


Read it against the chapter's two rescue conditions. The self-call goes strictly *downward* (`n - 1`). The descent has a *floor* (`n == 0`, where the procedure acts alone and returns). Both present: legitimate. Run it, then trace it in your head the slow way: `countdown(5)` prints 5 and asks `countdown(4)` to do the rest, which prints 4 and asks `countdown(3)`... Every call trusts the smaller call completely. That trust is the whole technique.

## 2. Hanoi in seven lines

The chapter's three-line plan — clear, cross, restack — translated nearly word for word:

In [2]:
def hanoi(n, source, target, spare):
    if n == 1:
        print("disk 1:", source, "->", target)
        return
    hanoi(n - 1, source, spare, target)      # clear the way (note: spare is the goal)
    print("disk", n, ":", source, "->", target)   # the big crossing
    hanoi(n - 1, spare, target, source)      # restack on top

hanoi(3, "A", "C", "B")

disk 1: A -> C
disk 2 : A -> B
disk 1: C -> B
disk 3 : A -> C
disk 1: B -> A
disk 2 : B -> C
disk 1: A -> C


Run it and check the seven moves against the solution your hands found with coins. Then the only part of this program that ever confuses anyone: the *peg arguments swap roles* at each call. In the first self-call, the spare peg is the destination — clearing the way means parking the small tower *out of* the way. Trace `hanoi(2, "A", "B", "C")` (just that call) on paper and confirm the pegs land where the plan demands.

How big a tower dares the machine? First predict from the chapter's table how many moves `hanoi(20, ...)` would print, then *don't print them* — counting wants a different version:

In [3]:
def hanoi_moves(n):
    if n == 1:
        return 1
    return hanoi_moves(n - 1) + 1 + hanoi_moves(n - 1)

print(hanoi_moves(10))
print(hanoi_moves(20))
print(hanoi_moves(30))

1023
1048575
1073741823


Check all three against 2ⁿ − 1. Then read the function once more: it is the recurrence M(n) = 2M(n−1) + 1, alive. The chapter's cost *equation* and this cost *program* are the same sentence in two grammars.

## 3. Making the staircase visible

Every call in §2 ran at some *depth* — Hanoi-for-3 paused while Hanoi-for-2 worked, which paused for Hanoi-for-1. The machine keeps the paused calls on its stack; let's print the stack's shape directly. Pass the depth along, and indent by it:

In [4]:
def hanoi_trace(n, source, target, spare, depth):
    print("  " * depth, "want", n, "disks:", source, "->", target)
    if n == 1:
        print("  " * depth, "move disk 1")
        return
    hanoi_trace(n - 1, source, spare, target, depth + 1)
    print("  " * depth, "move disk", n)
    hanoi_trace(n - 1, spare, target, source, depth + 1)

hanoi_trace(3, "A", "C", "B", 0)

 want 3 disks: A -> C
   want 2 disks: A -> B
     want 1 disks: A -> C
     move disk 1
   move disk 2
     want 1 disks: C -> B
     move disk 1
 move disk 3
   want 2 disks: B -> C
     want 1 disks: B -> A
     move disk 1
   move disk 2
     want 1 disks: A -> C
     move disk 1


(`"  " * depth` is a small Python kindness: a string times a number repeats it.) Run it and *read the left margin*: the indentation breathes in and out, deeper as work is delegated, shallower as delegations finish. That margin **is** the call stack — the deep-water box made of whitespace. Draw the same shape by hand for `hanoi_trace(2, ...)` before believing the machine's.

## 4. Merge sort, audited

The chapter sorted cards: split, sort each half somehow, merge. The merge first — it is where all real work happens:

In [5]:
def merge(left, right):
    result = []
    i = 0                      # finger on left pile
    j = 0                      # finger on right pile
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i = i + 1
        else:
            result.append(right[j])
            j = j + 1
    while i < len(left):       # one pile ran dry; sweep the other in
        result.append(left[i])
        i = i + 1
    while j < len(right):
        result.append(right[j])
        j = j + 1
    return result

print(merge([2, 9, 11, 30], [4, 5, 12, 28]))

[2, 4, 5, 9, 11, 12, 28, 30]


Two fingers, two piles, always take the smaller top card — the code is your hands from the chapter. Now the recursion around it, with one new piece of notation: `items[:mid]` is the list *up to* position mid, and `items[mid:]` is the rest. Splitting is the idea; the colon is just its spelling.

In [6]:
def merge_sort(items):
    if len(items) <= 1:        # the floor: tiny piles are born sorted
        return items
    mid = len(items) // 2
    left = merge_sort(items[:mid])
    right = merge_sort(items[mid:])
    return merge(left, right)

print(merge_sort([7, 3, 9, 1, 8, 2, 6, 4]))

[1, 2, 3, 4, 6, 7, 8, 9]


Eight lines around a merge. Test it on your own lists — include an empty list, a one-item list, a list with repeats, a list already sorted backward. (Testing the *edges* first is a habit worth stealing from professionals; Chapter 1's human moment is the reason the habit exists.)

## 5. The bill, audited

The chapter promised C(n) ≈ n log₂ n in the currency of element-steps — one step per card the merge touches. Send the count climbing back up the recursion alongside the sorted list; the function now returns a two-item list, `[sorted, steps]`:

In [ ]:
def merge_sort_count(items):
    if len(items) <= 1:
        return [items, 0]
    mid = len(items) // 2
    left = merge_sort_count(items[:mid])
    right = merge_sort_count(items[mid:])
    merged = merge(left[0], right[0])
    steps = left[1] + right[1] + len(merged)   # this merge touches every card once
    return [merged, steps]

Now the audit, in the style of Lab 1's race — and watch the *per-item* column:

In [ ]:
for n in [1000, 10000, 100000, 1000000]:
    items = list(range(n, 0, -1))          # n, n-1, ..., 1: sorted backward
    result = merge_sort_count(items)
    steps = result[1]
    print(n, steps, "steps;", steps / n, "per item")

The left column multiplies by 10; the per-item column climbs by a little over 3 each row. You have seen that gait before — it is the logarithm's, from Chapter 1's table, now appearing as the *per-item* price of sorting. Total steps ≈ n × (log₂ n-ish per item): the tree's level-count, measured in the wild. Check one row against arithmetic: for n = 1,000,000, is steps close to 20 million?

## Break it

Predict first. Run second. Explain any gap third.

**1.** Starve `countdown` of its floor: delete the `if n == 0:` block (keep the rest) and call `countdown(5)`. Predict precisely what the machine will do — forever? crash? something else? — then run it and *read the error to its last line*. The number it complains about (Python's recursion limit, near a thousand) is the deep-water box's claim that the stack is physical, demonstrated. How is this failure different in kind from Sevenia's in Lab 2 — which lab's program was *wrong*, and which was *exhausted*?

**2.** In `hanoi`, swap the two recursive calls (clear-the-way and restack trade places). Predict: will the printed moves still be *legal* — no large disk onto a small one? Then run it for n = 3 and *audit the move list with your coins*, move by move. Find the first illegal moment and describe which trust was betrayed.

**3.** In `merge_sort_count`, change `mid = len(items) // 2` to `mid = 1`: every split now peels a single card off the front instead of halving. Two predictions before running: is the output still *correct*? and what happens to the cost — per-item column flat-ish like the logarithm, or growing with n? Audit the lopsided version on sizes 200, 400, 800 (not the big sizes — you'll discover why in a moment) and look at the *total* steps as n doubles. Doubling the input does what to the bill? Connect it to the chapter's closing sentence: *where a recursion sends its work is the whole game*. You have just met, in your own code, the difference between n log n and n² — Chapter 6 makes it official. Then, deliberately, run size 2,000 and read the error to its last line: the lopsided tree is n calls deep where the balanced tree was log₂ n deep, and the machine's stack — Break-it 1's discovery — just refused the architecture, not the arithmetic.

**4.** In `merge`, change `<=` to `<`. Predict whether any list of *numbers* can ever come out unsorted. Run your tests; they should all pass. So the character changed nothing? To find out, sort things that can genuinely *tie*. Make each item a [score, name] pair and — one more edit — make the merge compare **scores only**: change the comparison line to `if left[i][0] <= right[j][0]:`. Now sort `[[3, "Ada"], [1, "Hua"], [3, "Noether"]]` with the `<=` version and the `<` version and compare the outputs. What happened to the two score-3 students, and which version keeps tied items in their original order? (Chapter 6 names this property; you have discovered it early. And a side puzzle: why did the experiment *need* the scores-only edit — what was Python quietly doing with the pairs before it?)



<div style="page-break-after: always;"></div>